# 📬 Email Engagement Recommender — Two-Tower Architecture
### Saks Global | CRM Personalization | PyTorch Implementation

---

## Notebook Overview

This notebook walks through the full pipeline of the Email Engagement Recommender built at Saks Global:

1. **Synthetic Data Generation** — Simulating Saks-like customer + campaign data
2. **Feature Engineering** — Building user-side and campaign-side feature vectors
3. **Temporal Train/Val/Test Split** — Avoiding data leakage
4. **Two-Tower Architecture in PyTorch** — User Tower + Campaign Tower + dot product scoring
5. **Training Loop** — With class imbalance handling, early stopping, MLflow-style tracking
6. **Offline Evaluation** — AUC-ROC, AUC-PR, NDCG@K, Precision@K
7. **Productionization Simulation** — Pre-computing user embeddings, scoring at send time
8. **A/B Test Simulation** — Comparing model vs. rule-based baseline

---

**Business Problem Recap:**  
For every `(customer, campaign)` pair — predict the probability of a click.  
Use that score to decide **who gets which campaign** — replacing legacy rule-based segmentation.


## Step 0 — Install & Import Dependencies

In [ ]:
# Install required libraries (run once)
# !pip install torch scikit-learn pandas numpy matplotlib seaborn tqdm

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import random
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Sklearn
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_score, recall_score, roc_curve,
    precision_recall_curve
)

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

---
## Step 1 — Synthetic Data Generation

We simulate a Saks-like dataset:
- **50,000 customers** with behavioral features
- **200 email campaigns** sent over 12 months
- Each customer may receive multiple campaigns
- Click labels generated with a realistic 8–12% CTR baseline

> **Key design choice:** Click probability is *correlated* with feature similarity between customer preferences and campaign content — so the model has real signal to learn.

In [ ]:
# ─────────────────────────────────────────────
# 1A. Generate Customer Master Table
# ─────────────────────────────────────────────

N_CUSTOMERS = 50_000
N_CAMPAIGNS = 200
N_INTERACTIONS = 300_000  # total (customer, campaign) send events

CATEGORIES = ['Shoes', 'Bags', 'Apparel', 'Beauty', 'Accessories', 'Home']
BRANDS = ['BrandA', 'BrandB', 'BrandC', 'BrandD', 'BrandE']
CAMPAIGN_TYPES = ['Promotional', 'Editorial', 'Reactivation', 'Loyalty', 'NewArrival']

print("Generating customer features...")

customers = pd.DataFrame({
    'customer_id': range(N_CUSTOMERS),
    
    # Email engagement history
    'open_rate_30d':      np.clip(np.random.beta(2, 5, N_CUSTOMERS), 0, 1),
    'ctr_30d':            np.clip(np.random.beta(1, 10, N_CUSTOMERS), 0, 1),
    'ctr_60d':            np.clip(np.random.beta(1, 9, N_CUSTOMERS), 0, 1),
    'ctr_90d':            np.clip(np.random.beta(1, 8, N_CUSTOMERS), 0, 1),
    
    # Unsubscribe / fatigue signals
    'unsub_flag':         np.random.binomial(1, 0.05, N_CUSTOMERS),   # 5% unsub rate
    'soft_bounce_rate':   np.clip(np.random.beta(1, 20, N_CUSTOMERS), 0, 1),
    'emails_received_30d': np.random.randint(1, 20, N_CUSTOMERS),
    
    # Purchase behavior
    'days_since_last_purchase': np.random.exponential(45, N_CUSTOMERS).astype(int),
    'orders_l12m':         np.random.poisson(3, N_CUSTOMERS),
    'avg_order_value':     np.random.lognormal(5.5, 0.8, N_CUSTOMERS),  # ~$250 mean
    'total_spend_l12m':    np.random.lognormal(6.5, 1.0, N_CUSTOMERS),
    
    # Category affinity (which category does this customer prefer?)
    # Represented as a dominant category index (0–5)
    'top_category_idx':    np.random.randint(0, len(CATEGORIES), N_CUSTOMERS),
    'category_diversity':  np.random.uniform(0, 1, N_CUSTOMERS),  # 0=mono, 1=diverse
    
    # Brand affinity
    'top_brand_idx':       np.random.randint(0, len(BRANDS), N_CUSTOMERS),
    
    # Price sensitivity (0=full price buyer, 1=discount-only)
    'price_sensitivity':   np.random.beta(2, 3, N_CUSTOMERS),
    
    # Channel preference
    'email_pref_score':    np.random.beta(3, 2, N_CUSTOMERS),  # higher = prefers email
    
    # Tenure
    'tenure_months':       np.random.randint(1, 60, N_CUSTOMERS),
    
    # Days since last email click
    'days_since_last_click': np.random.exponential(30, N_CUSTOMERS).astype(int),
})

print(f"Customer table: {customers.shape}")
customers.head(3)

In [ ]:
# ─────────────────────────────────────────────
# 1B. Generate Campaign Master Table
# ─────────────────────────────────────────────

print("Generating campaign features...")

# Campaigns are spread over 12 months (Jan–Dec 2023)
start_date = datetime(2023, 1, 1)
campaign_dates = [start_date + timedelta(days=random.randint(0, 364)) for _ in range(N_CAMPAIGNS)]

campaigns = pd.DataFrame({
    'campaign_id': range(N_CAMPAIGNS),
    'send_date': campaign_dates,
    
    # Campaign type
    'campaign_type_idx': np.random.randint(0, len(CAMPAIGN_TYPES), N_CAMPAIGNS),
    
    # Discount depth (0 = no discount, 1 = 100% off)
    'discount_depth': np.random.beta(2, 5, N_CAMPAIGNS),  # ~0.2–0.3 mean
    
    # Featured category
    'featured_category_idx': np.random.randint(0, len(CATEGORIES), N_CAMPAIGNS),
    
    # Featured brand
    'featured_brand_idx': np.random.randint(0, len(BRANDS), N_CAMPAIGNS),
    
    # Creative theme: 0=sale, 1=new arrival, 2=curated, 3=event
    'creative_theme': np.random.randint(0, 4, N_CAMPAIGNS),
    
    # Send timing
    'send_dow': np.random.randint(0, 7, N_CAMPAIGNS),   # day of week
    'send_hour': np.random.randint(6, 22, N_CAMPAIGNS), # hour of day
    
    # Historical avg CTR (from prior sends of same campaign type)
    'hist_avg_ctr': np.clip(np.random.beta(1.5, 12, N_CAMPAIGNS), 0.01, 0.3),
    
    # Subject line embedding (simplified: 8-dim random vector as proxy for BERT embedding)
    **{f'subject_emb_{i}': np.random.randn(N_CAMPAIGNS) for i in range(8)}
})

campaigns = campaigns.sort_values('send_date').reset_index(drop=True)
print(f"Campaign table: {campaigns.shape}")
campaigns.head(3)

In [ ]:
# ─────────────────────────────────────────────
# 1C. Generate Interaction Table (Send Events)
# ─────────────────────────────────────────────
# 
# Each row = one (customer, campaign) send event
# Click label is generated with realistic signal:
#   - Customers with matching category/brand affinity click MORE
#   - High CTR history → more likely to click
#   - Unsubscribed customers → never click
#   - High fatigue (many emails received) → click less

print("Generating interaction events (this may take a moment)...")

cust_ids = np.random.randint(0, N_CUSTOMERS, N_INTERACTIONS)
camp_ids = np.random.randint(0, N_CAMPAIGNS, N_INTERACTIONS)

# Pull relevant features for label generation
cust_ctr     = customers['ctr_30d'].values[cust_ids]
cust_cat     = customers['top_category_idx'].values[cust_ids]
cust_brand   = customers['top_brand_idx'].values[cust_ids]
cust_unsub   = customers['unsub_flag'].values[cust_ids]
cust_fatigue = customers['emails_received_30d'].values[cust_ids]
cust_price   = customers['price_sensitivity'].values[cust_ids]

camp_cat     = campaigns['featured_category_idx'].values[camp_ids]
camp_brand   = campaigns['featured_brand_idx'].values[camp_ids]
camp_disc    = campaigns['discount_depth'].values[camp_ids]
camp_ctr     = campaigns['hist_avg_ctr'].values[camp_ids]

# ── Click Probability Formula ──────────────────────────────────────
# Base: customer's historical CTR
# Boost: category match, brand match, discount × price sensitivity
# Penalty: unsub flag, high email fatigue

category_match = (cust_cat == camp_cat).astype(float)   # 1 if same category
brand_match    = (cust_brand == camp_brand).astype(float)
fatigue_penalty = np.clip(cust_fatigue / 20.0, 0, 1)    # normalized fatigue
discount_affinity = camp_disc * cust_price               # discount × price sensitivity

click_prob = (
    0.04                              # base rate
    + 0.20 * cust_ctr                 # personal engagement history
    + 0.10 * category_match           # category relevance
    + 0.08 * brand_match              # brand relevance
    + 0.06 * discount_affinity        # discount affinity
    + 0.05 * camp_ctr                 # campaign quality
    - 0.10 * fatigue_penalty          # fatigue penalty
    - 1.00 * cust_unsub               # hard zero for unsubscribed
)

click_prob = np.clip(click_prob, 0, 1)
clicks = np.random.binomial(1, click_prob)

# Build interaction dataframe
interactions = pd.DataFrame({
    'customer_id': cust_ids,
    'campaign_id': camp_ids,
    'clicked': clicks,
})

# Merge send_date from campaigns for temporal split
interactions = interactions.merge(
    campaigns[['campaign_id', 'send_date']], on='campaign_id'
).sort_values('send_date').reset_index(drop=True)

print(f"Interactions table: {interactions.shape}")
print(f"Overall CTR: {interactions['clicked'].mean():.3f} ({interactions['clicked'].mean()*100:.1f}%)")
interactions.head(3)

---
## Step 2 — Feature Engineering

We build **two separate feature matrices**:
- `user_features`: One row per customer — all behavioral signals
- `campaign_features`: One row per campaign — all content signals

These feed into their respective towers **independently**.

In [ ]:
# ─────────────────────────────────────────────
# 2A. User Feature Matrix
# ─────────────────────────────────────────────

USER_FEATURE_COLS = [
    'open_rate_30d', 'ctr_30d', 'ctr_60d', 'ctr_90d',
    'soft_bounce_rate', 'emails_received_30d',
    'days_since_last_purchase', 'orders_l12m',
    'avg_order_value', 'total_spend_l12m',
    'top_category_idx', 'category_diversity',
    'top_brand_idx', 'price_sensitivity',
    'email_pref_score', 'tenure_months',
    'days_since_last_click',
    # NOTE: unsub_flag excluded from features — it's used as a hard business rule
    # at send time, not a model feature (to avoid the model learning to ignore it)
]

user_feature_matrix = customers[USER_FEATURE_COLS].copy()

# Normalize continuous features
user_scaler = StandardScaler()
user_feature_matrix_scaled = user_scaler.fit_transform(user_feature_matrix)

print(f"User feature matrix shape: {user_feature_matrix_scaled.shape}")
print(f"User feature dim (input to User Tower): {user_feature_matrix_scaled.shape[1]}")

In [ ]:
# ─────────────────────────────────────────────
# 2B. Campaign Feature Matrix
# ─────────────────────────────────────────────

CAMPAIGN_FEATURE_COLS = [
    'campaign_type_idx', 'discount_depth',
    'featured_category_idx', 'featured_brand_idx',
    'creative_theme', 'send_dow', 'send_hour',
    'hist_avg_ctr',
] + [f'subject_emb_{i}' for i in range(8)]  # subject line embedding dims

campaign_feature_matrix = campaigns[CAMPAIGN_FEATURE_COLS].copy()

campaign_scaler = StandardScaler()
campaign_feature_matrix_scaled = campaign_scaler.fit_transform(campaign_feature_matrix)

print(f"Campaign feature matrix shape: {campaign_feature_matrix_scaled.shape}")
print(f"Campaign feature dim (input to Campaign Tower): {campaign_feature_matrix_scaled.shape[1]}")

In [ ]:
# ─────────────────────────────────────────────
# 2C. Build the Full Interaction Feature Matrix
# ─────────────────────────────────────────────
# 
# Each row in interactions → lookup user features + campaign features
# This is what goes into the Dataset / DataLoader

# Convert to numpy for fast indexing
user_feat_np = user_feature_matrix_scaled.astype(np.float32)
camp_feat_np = campaign_feature_matrix_scaled.astype(np.float32)

print("Feature lookup arrays ready.")
print(f"  user_feat_np: {user_feat_np.shape}   → indexed by customer_id")
print(f"  camp_feat_np: {camp_feat_np.shape}  → indexed by campaign_id")

USER_DIM = user_feat_np.shape[1]
CAMP_DIM = camp_feat_np.shape[1]

---
## Step 3 — Temporal Train / Val / Test Split

**Critical:** We split by send date, NOT randomly.

```
|─── Train (Jan–Sep) ───|─── Val (Oct) ───|─── Test (Nov–Dec) ───|
```

This avoids **data leakage** — the model never sees future click patterns during training.

In [ ]:
# ─────────────────────────────────────────────
# 3. Temporal Split
# ─────────────────────────────────────────────

train_cutoff = datetime(2023, 10, 1)
val_cutoff   = datetime(2023, 11, 1)

train_df = interactions[interactions['send_date'] <  train_cutoff].reset_index(drop=True)
val_df   = interactions[(interactions['send_date'] >= train_cutoff) &
                         (interactions['send_date'] <  val_cutoff)].reset_index(drop=True)
test_df  = interactions[interactions['send_date'] >= val_cutoff].reset_index(drop=True)

for name, df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    ctr = df['clicked'].mean()
    print(f"{name:6s}: {len(df):>7,} rows | CTR: {ctr:.3f} | "
          f"Date range: {df['send_date'].min().date()} → {df['send_date'].max().date()}")

# Class imbalance ratio
neg = (train_df['clicked'] == 0).sum()
pos = (train_df['clicked'] == 1).sum()
pos_weight = neg / pos
print(f"\nClass imbalance ratio (neg/pos): {pos_weight:.1f}x → used as pos_weight in BCEWithLogitsLoss")

---
## Step 4 — PyTorch Dataset & DataLoader

In [ ]:
class EmailDataset(Dataset):
    """
    Each item returns:
      - user_feat:  feature vector for the customer (User Tower input)
      - camp_feat:  feature vector for the campaign (Campaign Tower input)
      - label:      1 = clicked, 0 = not clicked
    
    Note: We store only IDs in the dataframe and do lookup here.
    This keeps memory efficient — we don't duplicate feature arrays.
    """
    def __init__(self, df, user_feat_np, camp_feat_np):
        self.customer_ids = df['customer_id'].values
        self.campaign_ids = df['campaign_id'].values
        self.labels       = df['clicked'].values.astype(np.float32)
        self.user_feats   = user_feat_np
        self.camp_feats   = camp_feat_np

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        cust_id = self.customer_ids[idx]
        camp_id = self.campaign_ids[idx]
        return (
            torch.tensor(self.user_feats[cust_id], dtype=torch.float32),
            torch.tensor(self.camp_feats[camp_id], dtype=torch.float32),
            torch.tensor(self.labels[idx],         dtype=torch.float32),
        )


BATCH_SIZE = 2048

train_dataset = EmailDataset(train_df, user_feat_np, camp_feat_np)
val_dataset   = EmailDataset(val_df,   user_feat_np, camp_feat_np)
test_dataset  = EmailDataset(test_df,  user_feat_np, camp_feat_np)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)} | Test batches: {len(test_loader)}")

# Quick sanity check — shapes
u, c, y = next(iter(train_loader))
print(f"\nBatch shapes — User: {u.shape} | Campaign: {c.shape} | Label: {y.shape}")

---
## Step 5 — Two-Tower Model Architecture

```
USER TOWER                          CAMPAIGN TOWER
─────────────────────               ─────────────────────
[17-dim user features]              [16-dim campaign features]
         │                                    │
    Dense(256, ReLU)                    Dense(256, ReLU)
    BatchNorm + Dropout                 BatchNorm + Dropout
         │                                    │
    Dense(128, ReLU)                    Dense(128, ReLU)
    BatchNorm + Dropout                 BatchNorm + Dropout
         │                                    │
    Dense(64) → L2 normalize           Dense(64) → L2 normalize
         │                                    │
   User Embedding                   Campaign Embedding
         └──────────── Dot Product ───────────┘
                            │
                    Sigmoid → Click Prob
```

In [ ]:
class Tower(nn.Module):
    """
    A single tower (User or Campaign).
    
    Architecture:
        input_dim → 256 → 128 → embedding_dim
    
    Each hidden layer:
        Linear → BatchNorm → ReLU → Dropout
    
    Output layer:
        Linear → L2 normalization
        (normalization keeps embeddings on unit sphere → dot product = cosine similarity)
    """
    def __init__(self, input_dim: int, embedding_dim: int = 64, dropout: float = 0.3):
        super().__init__()
        
        self.net = nn.Sequential(
            # Layer 1: input → 256
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Layer 2: 256 → 128
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            
            # Output: 128 → embedding_dim (no activation — we normalize after)
            nn.Linear(128, embedding_dim),
        )
    
    def forward(self, x):
        emb = self.net(x)
        # L2 normalize → unit sphere → dot product becomes cosine similarity
        return F.normalize(emb, p=2, dim=-1)


class TwoTowerModel(nn.Module):
    """
    Full Two-Tower Model.
    
    - user_tower:     maps user features → 64-d embedding
    - campaign_tower: maps campaign features → 64-d embedding
    - score:          dot product of the two embeddings → logit
    
    We scale the dot product by a learnable temperature τ.
    This is standard in contrastive / retrieval models (e.g., CLIP, DPR).
    Higher temperature = softer probabilities.
    """
    def __init__(
        self,
        user_dim: int,
        campaign_dim: int,
        embedding_dim: int = 64,
        dropout: float = 0.3
    ):
        super().__init__()
        self.user_tower     = Tower(user_dim,     embedding_dim, dropout)
        self.campaign_tower = Tower(campaign_dim, embedding_dim, dropout)
        
        # Learnable temperature scaling
        self.temperature = nn.Parameter(torch.ones(1) * 10.0)
    
    def forward(self, user_feat, camp_feat):
        user_emb = self.user_tower(user_feat)       # (B, 64)
        camp_emb = self.campaign_tower(camp_feat)   # (B, 64)
        
        # Dot product of L2-normalized vectors = cosine similarity ∈ [-1, 1]
        # Multiply by temperature to get a proper logit scale
        dot = (user_emb * camp_emb).sum(dim=-1)     # (B,)
        logit = dot * self.temperature
        return logit  # raw logit — sigmoid applied in loss / inference
    
    def get_user_embedding(self, user_feat):
        """Used at productionization: pre-compute user embeddings nightly."""
        return self.user_tower(user_feat)
    
    def get_campaign_embedding(self, camp_feat):
        """Used at productionization: pre-compute campaign embeddings at campaign creation."""
        return self.campaign_tower(camp_feat)
    
    def score_from_embeddings(self, user_emb, camp_emb):
        """Fast scoring at send time: just dot product (no tower forward pass)."""
        dot = (user_emb * camp_emb).sum(dim=-1)
        return torch.sigmoid(dot * self.temperature)


# Instantiate model
EMBEDDING_DIM = 64
model = TwoTowerModel(
    user_dim=USER_DIM,
    campaign_dim=CAMP_DIM,
    embedding_dim=EMBEDDING_DIM,
    dropout=0.3
).to(device)

# Print model summary
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTotal trainable parameters: {total_params:,}")

---
## Step 6 — Training Loop

Key decisions:
- **Loss:** `BCEWithLogitsLoss` with `pos_weight` to handle class imbalance (9:1 neg:pos)
- **Optimizer:** Adam with weight decay
- **Scheduler:** ReduceLROnPlateau on validation AUC
- **Early stopping:** Stop if val AUC doesn't improve for 3 epochs

In [ ]:
# ─────────────────────────────────────────────
# Loss, Optimizer, Scheduler
# ─────────────────────────────────────────────

# pos_weight: upweights the positive (click) class in the loss
# = num_negatives / num_positives ≈ 9x in our data
pos_weight_tensor = torch.tensor([pos_weight], dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)

optimizer = Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4  # L2 regularization
)

scheduler = ReduceLROnPlateau(
    optimizer,
    mode='max',      # maximize val AUC
    patience=2,
    factor=0.5,
    verbose=True
)

print(f"Loss function: BCEWithLogitsLoss (pos_weight={pos_weight:.2f})")
print(f"Optimizer: Adam (lr=1e-3, weight_decay=1e-4)")
print(f"Scheduler: ReduceLROnPlateau (patience=2, factor=0.5)")

In [ ]:
# ─────────────────────────────────────────────
# Helper: Evaluate on a DataLoader
# ─────────────────────────────────────────────

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_labels, all_probs = [], []
    
    with torch.no_grad():
        for user_feat, camp_feat, labels in loader:
            user_feat = user_feat.to(device)
            camp_feat = camp_feat.to(device)
            labels    = labels.to(device)
            
            logits = model(user_feat, camp_feat)
            loss   = criterion(logits, labels)
            probs  = torch.sigmoid(logits)
            
            total_loss += loss.item() * len(labels)
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
    
    avg_loss = total_loss / len(loader.dataset)
    auc_roc  = roc_auc_score(all_labels, all_probs)
    auc_pr   = average_precision_score(all_labels, all_probs)
    
    return avg_loss, auc_roc, auc_pr, np.array(all_labels), np.array(all_probs)

In [ ]:
# ─────────────────────────────────────────────
# Training Loop with Early Stopping
# ─────────────────────────────────────────────

N_EPOCHS = 15
PATIENCE = 3       # early stopping patience

best_val_auc = 0
patience_counter = 0
history = []

print(f"{'Epoch':>5} | {'Train Loss':>10} | {'Val Loss':>8} | {'Val AUC-ROC':>11} | {'Val AUC-PR':>10} | {'LR':>8}")
print("-" * 70)

for epoch in range(1, N_EPOCHS + 1):
    # ── Train ──────────────────────────────────
    model.train()
    train_loss = 0
    
    for user_feat, camp_feat, labels in train_loader:
        user_feat = user_feat.to(device)
        camp_feat = camp_feat.to(device)
        labels    = labels.to(device)
        
        optimizer.zero_grad()
        logits = model(user_feat, camp_feat)
        loss   = criterion(logits, labels)
        loss.backward()
        
        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        train_loss += loss.item() * len(labels)
    
    train_loss /= len(train_loader.dataset)
    
    # ── Validate ───────────────────────────────
    val_loss, val_auc_roc, val_auc_pr, _, _ = evaluate(model, val_loader, criterion, device)
    
    # ── Scheduler step ────────────────────────
    scheduler.step(val_auc_roc)
    current_lr = optimizer.param_groups[0]['lr']
    
    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_auc_roc': val_auc_roc,
        'val_auc_pr': val_auc_pr,
    })
    
    print(f"{epoch:>5} | {train_loss:>10.4f} | {val_loss:>8.4f} | "
          f"{val_auc_roc:>11.4f} | {val_auc_pr:>10.4f} | {current_lr:>8.6f}")
    
    # ── Early stopping ────────────────────────
    if val_auc_roc > best_val_auc:
        best_val_auc = val_auc_roc
        patience_counter = 0
        torch.save(model.state_dict(), '/tmp/best_two_tower.pt')
        print(f"         ✓ New best val AUC: {best_val_auc:.4f} — model saved")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            break

# Load best model
model.load_state_dict(torch.load('/tmp/best_two_tower.pt'))
print(f"\nBest val AUC-ROC: {best_val_auc:.4f}")

---
## Step 7 — Training Curves

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Two-Tower Training Curves — Saks Email Engagement Recommender', fontsize=13, y=1.02)

# Loss
axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train', marker='o')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val',   marker='s')
axes[0].set_title('Loss (BCEWithLogitsLoss)')
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# AUC-ROC
axes[1].plot(hist_df['epoch'], hist_df['val_auc_roc'], color='green', marker='o')
axes[1].set_title('Validation AUC-ROC')
axes[1].set_xlabel('Epoch'); axes[1].set_ylim([0.5, 1.0]); axes[1].grid(True, alpha=0.3)

# AUC-PR
axes[2].plot(hist_df['epoch'], hist_df['val_auc_pr'], color='orange', marker='o')
axes[2].set_title('Validation AUC-PR\n(more meaningful under imbalance)')
axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 8 — Offline Evaluation on Test Set

Evaluate the champion model on the held-out Nov–Dec test set.

In [ ]:
# ─────────────────────────────────────────────
# Test Set Evaluation
# ─────────────────────────────────────────────

test_loss, test_auc_roc, test_auc_pr, y_true, y_prob = evaluate(
    model, test_loader, criterion, device
)

print("=" * 45)
print("TEST SET EVALUATION (Nov–Dec holdout)")
print("=" * 45)
print(f"  AUC-ROC:   {test_auc_roc:.4f}")
print(f"  AUC-PR:    {test_auc_pr:.4f}")
print(f"  Test Loss: {test_loss:.4f}")

In [ ]:
# ─────────────────────────────────────────────
# Precision@K and NDCG@K
# ─────────────────────────────────────────────
# 
# In a send-decisioning context:
# We rank all campaigns for a customer and pick Top-K.
# We want: the campaigns we send to actually be the ones that get clicked.

def precision_at_k(y_true, y_prob, k):
    """Precision@K: of top-K scored, what fraction was actually clicked."""
    top_k_idx = np.argsort(y_prob)[::-1][:k]
    return y_true[top_k_idx].mean()

def ndcg_at_k(y_true, y_prob, k):
    """NDCG@K: ranking quality metric — penalizes relevant items ranked lower."""
    top_k_idx = np.argsort(y_prob)[::-1][:k]
    gains = y_true[top_k_idx]
    discounts = np.log2(np.arange(2, k + 2))  # [log2(2), log2(3), ...]
    dcg = (gains / discounts).sum()
    # Ideal DCG: all positives ranked at the top
    ideal = np.sort(y_true)[::-1][:k]
    idcg = (ideal / discounts[:len(ideal)]).sum()
    return dcg / idcg if idcg > 0 else 0.0


print("Ranking Metrics on Test Set:")
print("-" * 30)
for k in [10, 20, 50, 100]:
    p_at_k = precision_at_k(y_true, y_prob, k)
    n_at_k = ndcg_at_k(y_true, y_prob, k)
    print(f"  K={k:>3}  |  Precision@K: {p_at_k:.4f}  |  NDCG@K: {n_at_k:.4f}")

In [ ]:
# ─────────────────────────────────────────────
# ROC Curve + PR Curve
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Test Set — ROC Curve & Precision-Recall Curve', fontsize=13)

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_prob)
axes[0].plot(fpr, tpr, color='steelblue', lw=2, label=f'Two-Tower (AUC = {test_auc_roc:.3f})')
axes[0].plot([0,1],[0,1], 'k--', lw=1, label='Random baseline')
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# PR Curve
prec, rec, _ = precision_recall_curve(y_true, y_prob)
baseline_pr = y_true.mean()
axes[1].plot(rec, prec, color='darkorange', lw=2, label=f'Two-Tower (AUC-PR = {test_auc_pr:.3f})')
axes[1].axhline(baseline_pr, color='k', linestyle='--', lw=1, label=f'Random (CTR={baseline_pr:.3f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve\n(more meaningful under class imbalance)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/tmp/eval_curves.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 9 — Productionization Simulation

This is the key architectural advantage of the two-tower model:

1. **Nightly batch:** Pre-compute all user embeddings → store in cache
2. **Campaign creation:** Compute campaign embedding once
3. **Send time:** Just do dot product (no tower forward pass) — very fast

This allows scoring **millions of customers** at send time with minimal latency.

In [ ]:
# ─────────────────────────────────────────────
# STEP A: Nightly Batch — Pre-compute ALL user embeddings
# ─────────────────────────────────────────────

model.eval()
print("[NIGHTLY BATCH] Pre-computing user embeddings for all 50,000 customers...")

all_user_embeddings = {}  # In production: Redis / Feature Store

user_tensor = torch.tensor(user_feat_np, dtype=torch.float32)
EMBED_BATCH = 4096

with torch.no_grad():
    for start in range(0, N_CUSTOMERS, EMBED_BATCH):
        end = min(start + EMBED_BATCH, N_CUSTOMERS)
        batch = user_tensor[start:end].to(device)
        embs  = model.get_user_embedding(batch)  # (batch, 64)
        for i, cid in enumerate(range(start, end)):
            all_user_embeddings[cid] = embs[i].cpu().numpy()

print(f"  Done. Cached {len(all_user_embeddings):,} user embeddings (each 64-d)")
print(f"  Memory footprint: ~{len(all_user_embeddings) * 64 * 4 / 1024 / 1024:.1f} MB")

In [ ]:
# ─────────────────────────────────────────────
# STEP B: At Campaign Scheduling — Compute Campaign Embedding
# ─────────────────────────────────────────────

# Simulate: A new campaign (campaign_id=5) is being scheduled for tomorrow
target_campaign_id = 5

camp_features = torch.tensor(
    camp_feat_np[target_campaign_id], dtype=torch.float32
).unsqueeze(0).to(device)  # (1, 16)

with torch.no_grad():
    campaign_embedding = model.get_campaign_embedding(camp_features)  # (1, 64)

print(f"Campaign embedding computed for Campaign ID={target_campaign_id}")
print(f"  Shape: {campaign_embedding.shape} | Norm: {campaign_embedding.norm().item():.4f} (should be ~1.0)")

In [ ]:
# ─────────────────────────────────────────────
# STEP C: At Send Time — Score all customers for this campaign
# (dot product only — no tower forward pass)
# ─────────────────────────────────────────────

# Eligible send universe: exclude unsubscribed customers (hard business rule)
eligible_customers = customers[customers['unsub_flag'] == 0]['customer_id'].values
print(f"Eligible send universe: {len(eligible_customers):,} customers "
      f"(excluded {N_CUSTOMERS - len(eligible_customers):,} unsubscribed)")

# Stack all eligible user embeddings
user_embs_matrix = np.stack([all_user_embeddings[cid] for cid in eligible_customers])
user_embs_tensor = torch.tensor(user_embs_matrix, dtype=torch.float32).to(device)

camp_emb_broadcast = campaign_embedding.expand(len(eligible_customers), -1)  # (N, 64)

with torch.no_grad():
    scores = model.score_from_embeddings(user_embs_tensor, camp_emb_broadcast)  # (N,)

scores_np = scores.cpu().numpy()

print(f"\nScoring complete for {len(eligible_customers):,} customers")
print(f"  Score distribution: min={scores_np.min():.4f} | mean={scores_np.mean():.4f} | max={scores_np.max():.4f}")

# Apply send threshold
SEND_THRESHOLD = 0.15
send_list = eligible_customers[scores_np >= SEND_THRESHOLD]
print(f"\n  Send threshold: {SEND_THRESHOLD}")
print(f"  Customers to send: {len(send_list):,} ({len(send_list)/len(eligible_customers)*100:.1f}% of eligible)")
print(f"  → This list would be passed to the ESP (Email Service Provider)")

In [ ]:
# Score distribution visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Send-Time Scoring — Campaign 5', fontsize=12)

axes[0].hist(scores_np, bins=60, color='steelblue', edgecolor='white', linewidth=0.3)
axes[0].axvline(SEND_THRESHOLD, color='red', linestyle='--', linewidth=2, label=f'Send threshold = {SEND_THRESHOLD}')
axes[0].set_xlabel('Predicted Click Probability'); axes[0].set_ylabel('# Customers')
axes[0].set_title('Score Distribution Across All Eligible Customers')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Score decile analysis
score_df = pd.DataFrame({'customer_id': eligible_customers, 'score': scores_np})
score_df['decile'] = pd.qcut(score_df['score'], q=10, labels=False)
decile_stats = score_df.groupby('decile')['score'].agg(['mean', 'count'])

axes[1].bar(decile_stats.index + 1, decile_stats['mean'], color='steelblue', edgecolor='white')
axes[1].set_xlabel('Score Decile (1=lowest, 10=highest)')
axes[1].set_ylabel('Avg Predicted Score')
axes[1].set_title('Average Predicted Score by Decile')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/send_scoring.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 10 — A/B Test Simulation

Simulating the A/B test design used to validate the Two-Tower model against the legacy rule-based approach.

**Design:**
- Control (20%): Legacy rule-based selection — all customers who purchased in last 60 days
- Treatment (80%): Two-tower model — top-scored customers above threshold
- Primary metric: CTR uplift
- Guard rail: Unsubscribe rate

In [ ]:
from scipy import stats

# ─────────────────────────────────────────────
# A/B Test Setup
# ─────────────────────────────────────────────

np.random.seed(SEED)

# Customer-level randomization using hash-style assignment
# Same customer always in same bucket across sends
ab_assignment = customers['customer_id'].apply(
    lambda cid: 'control' if (cid % 100) < 20 else 'treatment'
)
customers['ab_group'] = ab_assignment

print(f"A/B split:")
print(customers['ab_group'].value_counts().to_string())

# ── Control: rule-based selection ──────────────
# Rule: send to all customers who purchased in last 60 days
control_customers = customers[
    (customers['ab_group'] == 'control') &
    (customers['days_since_last_purchase'] <= 60) &
    (customers['unsub_flag'] == 0)
].copy()

# ── Treatment: model-scored selection ──────────
treatment_score_df = pd.DataFrame({
    'customer_id': eligible_customers,
    'score': scores_np
})
treatment_score_df = treatment_score_df.merge(
    customers[['customer_id', 'ab_group']], on='customer_id'
)
treatment_customers = treatment_score_df[
    (treatment_score_df['ab_group'] == 'treatment') &
    (treatment_score_df['score'] >= SEND_THRESHOLD)
].copy()

print(f"\nControl send list (rule-based):  {len(control_customers):,} customers")
print(f"Treatment send list (model):     {len(treatment_customers):,} customers")

In [ ]:
# ─────────────────────────────────────────────
# Simulate Post-Send Outcomes
# ─────────────────────────────────────────────

# Control: clicks at ~8% base rate (rule-based is not very targeted)
ctrl_n    = len(control_customers)
ctrl_ctr  = 0.082  # legacy CTR
ctrl_clicks = np.random.binomial(ctrl_n, ctrl_ctr)
ctrl_unsub  = np.random.binomial(ctrl_n, 0.015)  # 1.5% unsub rate

# Treatment: model selects higher-propensity users → ~30% CTR uplift
trt_n    = len(treatment_customers)
trt_ctr  = ctrl_ctr * 1.30   # ~30% relative uplift
trt_clicks = np.random.binomial(trt_n, trt_ctr)
trt_unsub  = np.random.binomial(trt_n, 0.012)    # lower unsub (better targeting)

# ── Statistical significance test ─────────────
ctrl_obs_ctr = ctrl_clicks / ctrl_n
trt_obs_ctr  = trt_clicks / trt_n

# Two-proportion z-test
p_pool = (ctrl_clicks + trt_clicks) / (ctrl_n + trt_n)
se = np.sqrt(p_pool * (1 - p_pool) * (1/ctrl_n + 1/trt_n))
z_stat = (trt_obs_ctr - ctrl_obs_ctr) / se
p_value = 1 - stats.norm.cdf(z_stat)  # one-tailed

relative_uplift = (trt_obs_ctr - ctrl_obs_ctr) / ctrl_obs_ctr * 100

print("=" * 55)
print("A/B TEST RESULTS")
print("=" * 55)
print(f"{'Metric':<25} {'Control':>12} {'Treatment':>12}")
print("-" * 55)
print(f"{'Send Volume':<25} {ctrl_n:>12,} {trt_n:>12,}")
print(f"{'Clicks':<25} {ctrl_clicks:>12,} {trt_clicks:>12,}")
print(f"{'CTR':<25} {ctrl_obs_ctr:>12.4f} {trt_obs_ctr:>12.4f}")
print(f"{'Unsubscribes':<25} {ctrl_unsub:>12,} {trt_unsub:>12,}")
print(f"{'Unsub Rate':<25} {ctrl_unsub/ctrl_n:>12.4f} {trt_unsub/trt_n:>12.4f}")
print("-" * 55)
print(f"\nRelative CTR Uplift:  {relative_uplift:+.1f}%")
print(f"Z-statistic:          {z_stat:.3f}")
print(f"P-value:              {p_value:.6f}")
print(f"Significant (α=0.05): {'✓ YES' if p_value < 0.05 else '✗ NO'}")
print(f"\nGuard rail (unsub):   {'✓ PASS' if trt_unsub/trt_n < ctrl_unsub/ctrl_n * 1.1 else '✗ FAIL (unsub exceeded threshold)'}")

In [ ]:
# ─────────────────────────────────────────────
# A/B Result Visualization
# ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('A/B Test Results — Two-Tower vs. Rule-Based', fontsize=13)

# CTR comparison
groups = ['Control\n(Rule-Based)', 'Treatment\n(Two-Tower)']
ctrs   = [ctrl_obs_ctr * 100, trt_obs_ctr * 100]
colors = ['#95a5a6', '#2ecc71']

bars = axes[0].bar(groups, ctrs, color=colors, width=0.5, edgecolor='white')
axes[0].set_ylabel('Click-Through Rate (%)')
axes[0].set_title(f'CTR Comparison\n(+{relative_uplift:.1f}% relative uplift, p={p_value:.4f})')
for bar, val in zip(bars, ctrs):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.2f}%', ha='center', fontweight='bold', fontsize=11)
axes[0].set_ylim(0, max(ctrs) * 1.25); axes[0].grid(True, alpha=0.3, axis='y')

# Unsub rate comparison
unsubs = [ctrl_unsub/ctrl_n * 100, trt_unsub/trt_n * 100]
colors2 = ['#e74c3c', '#3498db']
bars2 = axes[1].bar(groups, unsubs, color=colors2, width=0.5, edgecolor='white')
axes[1].set_ylabel('Unsubscribe Rate (%)')
axes[1].set_title('Unsubscribe Rate\n(guard rail metric — must not increase)')
for bar, val in zip(bars2, unsubs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                 f'{val:.3f}%', ha='center', fontweight='bold', fontsize=11)
axes[1].set_ylim(0, max(unsubs) * 1.35); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('/tmp/ab_test_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Step 11 — Embedding Space Visualization

Sanity check: do user embeddings cluster meaningfully by category affinity?

In [ ]:
from sklearn.decomposition import PCA

# Sample 2,000 customers for visualization
SAMPLE_N = 2000
sample_ids = np.random.choice(N_CUSTOMERS, SAMPLE_N, replace=False)

sample_feats = torch.tensor(user_feat_np[sample_ids], dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    sample_embs = model.get_user_embedding(sample_feats).cpu().numpy()  # (2000, 64)

# PCA to 2D
pca = PCA(n_components=2, random_state=SEED)
embs_2d = pca.fit_transform(sample_embs)

sample_categories = customers['top_category_idx'].values[sample_ids]

plt.figure(figsize=(10, 7))
scatter = plt.scatter(
    embs_2d[:, 0], embs_2d[:, 1],
    c=sample_categories, cmap='tab10',
    alpha=0.5, s=15, linewidths=0
)
legend_labels = [f'Cat: {CATEGORIES[i]}' for i in range(len(CATEGORIES))]
plt.legend(handles=scatter.legend_elements()[0], labels=legend_labels,
           loc='upper right', fontsize=9)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.title('User Embedding Space (PCA 2D)\n'
          'Meaningful clustering = model learned category-specific preference representations')
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('/tmp/embedding_space.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"PCA explained variance: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, "
      f"PC2={pca.explained_variance_ratio_[1]*100:.1f}%")

---
## Step 12 — Summary: What Would You Say in the Interview

```
BUSINESS PROBLEM
  Legacy rule-based segmentation → wrong campaigns to wrong customers
  → high unsubscribes, low CTR, CRM health degradation

DATA
  User Tower inputs:     17 features — engagement history, purchase RFM,
                         category/brand affinity, price sensitivity, fatigue signals
  Campaign Tower inputs: 16 features — type, discount depth, category,
                         brand, creative theme, timing, subject line embedding
  Label:                 Binary click from historical send logs
  Scale:                 300K+ (customer × campaign) interactions over 12 months

ARCHITECTURE
  Two-Tower Deep Learning:
    User Tower:     17 → Dense(256) → Dense(128) → Embedding(64)
    Campaign Tower: 16 → Dense(256) → Dense(128) → Embedding(64)
    Scoring:        Dot product of L2-normalized embeddings × temperature → sigmoid

TRAINING
  Loss:      BCEWithLogitsLoss with pos_weight for class imbalance (~9:1)
  Optimizer: Adam + weight decay + gradient clipping
  Split:     Temporal (train: Jan-Sep | val: Oct | test: Nov-Dec)

PRODUCTIONIZATION
  Nightly: Pre-compute all user embeddings → cache
  Send time: Campaign embedding (once) + dot product per customer → O(N) not O(N×C)
  MLflow: Model registry, experiment tracking, champion/challenger versioning
  Monitoring: AUC drift + PSI on input features for retraining triggers

A/B TEST
  Design:  Customer-level randomization, 20/80 split, pre-committed MDE
  Primary metric: CTR uplift → ~30% relative improvement, p < 0.05
  Guard rail: Unsubscribe rate → flat or lower (model suppresses fatigued users)

WAYFAIR EXTENSION
  Add cross-channel layer: Email + Push + SMS unified scoring
  Add multi-objective: CTR vs. Revenue vs. Fatigue in a constrained optimization
  Add candidate retrieval: FAISS ANN search on user embeddings at Wayfair scale
```